"Text Document Classification Dataset"

In [14]:
#load the required python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [3]:
#load the data into dataframe
df = pd.read_csv("/content/df_file (1).csv")
df.head()

,Text,Label
0,Budget to set scene for election\n \n Gordon B...,0
1,Army chiefs in regiments decision\n \n Militar...,0
2,Howard denies split over ID cards\n \n Michael...,0
3,Observers to monitor UK election\n \n Minister...,0
4,Kilroy names election seat target\n \n Ex-chat...,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Text    2225 non-null   object
 1   Label   2225 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 34.9+ KB


In [6]:
df.shape

(2225, 2)

In [7]:
#Missing values check
df.isnull().sum()

,0
Text,0
Label,0


Exploratory Data Analysis

In [8]:
temp=df
temp['Label']= temp['Label'].map({0:'Politics', 1:'Sport', 3:'Entertainemnt',4:'Business'})
temp

,Text,Label
0,Budget to set scene for election\n \n Gordon B...,Politics
1,Army chiefs in regiments decision\n \n Militar...,Politics
2,Howard denies split over ID cards\n \n Michael...,Politics
3,Observers to monitor UK election\n \n Minister...,Politics
4,Kilroy names election seat target\n \n Ex-chat...,Politics
...,...,...
2220,India opens skies to competition\n \n India wi...,Business
2221,Yukos bankruptcy 'not US matter'\n \n Russian ...,Business
2222,Survey confirms property slowdown\n \n Governm...,Business
2223,High fuel prices hit BA's profits\n \n British...,Business


In [9]:
label_counts = temp["Label"].value_counts()
label_counts

,count
Label,
Sport,511
Business,510
Politics,417
Entertainemnt,386


In [10]:
import plotly.graph_objects as go

#Count labels
label_counts = temp["Label"].value_counts()

#Create pie chart
fig = go.Figure(go.Pie(
    labels=label_counts.index,
    values=label_counts,
    hole=0.5,
    marker=dict(colors=['#FF5733','#33FF57', '#3357FF'])
))

fig.update_layout(
    title="Distribution of Labels",
    width = 1200,
    template='plotly_white',
    annotations=[dict(text='Label', x=0.5, y=0.5, showarrow=False, font_size=12, font_color="white")]
)

fig.show()

In [11]:
temp

,Text,Label
0,Budget to set scene for election\n \n Gordon B...,Politics
1,Army chiefs in regiments decision\n \n Militar...,Politics
2,Howard denies split over ID cards\n \n Michael...,Politics
3,Observers to monitor UK election\n \n Minister...,Politics
4,Kilroy names election seat target\n \n Ex-chat...,Politics
...,...,...
2220,India opens skies to competition\n \n India wi...,Business
2221,Yukos bankruptcy 'not US matter'\n \n Russian ...,Business
2222,Survey confirms property slowdown\n \n Governm...,Business
2223,High fuel prices hit BA's profits\n \n British...,Business


In [15]:
fig=px.histogram(data_frame=temp, x='Label', marginal='violin')
fig.update_layout(width=1200, title='Distribution of Labels', template='plotly_dark')
fig.show()

Text Cleaning and Preprocessing

In [16]:
import re       # Regular Expressions
import nltk     # Natural language processing
from nltk.corpus import stopwords  # Stopwords
nltk.download('stopwords')  # Download stopwords corpus
# Get the stopwords for the language
stpwrds= stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [20]:
# General transformation in the text
def transformation(df, mc,):

    df[mc] = df[mc].replace("\n"," ").replace("\t"," ")
    df[mc] = df[mc].str.lower()
    df[mc] = df[mc].apply(lambda x: re.sub('@[^\s]+', '', x))  #Removes Twitter-style mentions (e.g., @username) from the text.
    df[mc] = df[mc].apply(lambda x: re.sub(r'\B#\S+', '', x))   #Removes hashtags (e.g., #hashtag) from the text.
    df[mc] = df[mc].apply(lambda x: re.sub(r"http\S+", "", x))  #Removes URLs from the text.
    df[mc] = df[mc].apply(lambda x: ' '.join(re.findall(r'\w+', x)))  #Keeps only alphanumeric characters (words) and removes everything else.
    df[mc] = df[mc].apply(lambda x: re.sub(r'\s+[b-zA-Z]\s+', ' ', x)) #Removes single characters (letters) that are surrounded by spaces, which are often
                                                                          # insignificant in text analysis.
    df[mc] = df[mc].apply(lambda x: re.sub(r'\s+', ' ', x, flags=re.I))  #Replaces multiple whitespace characters (spaces, tabs) with a single space.

    df[mc] = df[mc].apply(lambda x: ' '.join([word for word in x.split() if word not in stpwrds]))  #Removes stop words
                                        #(common words that may not add significant meaning) from the text. stpwrds is assumed to be a predefined list of stop words.

    df['words'] = df[mc].apply(lambda x: re.findall(r'\w+', x)) #Creates a new column words that contains lists of words extracted from the text.
    df['words_count'] = df.words.apply(len)  #Calculates the number of words in each text and stores it in a new column words_count.
    df['length'] = df[mc].apply(len)  #Computes the length of the text (number of characters) and stores it in a new column length.

    return df

In [24]:
data = transformation(df,'Text')
data.head()

,Text,Label,words,length,words_count
0,budget set scene election gordon brown seek pu...,Politics,"[budget, set, scene, election, gordon, brown, ...",2231,330
1,army chiefs regiments decision military chiefs...,Politics,"[army, chiefs, regiments, decision, military, ...",2078,273
2,howard denies split id cards michael howard de...,Politics,"[howard, denies, split, id, cards, michael, ho...",2178,316
3,observers monitor uk election ministers invite...,Politics,"[observers, monitor, uk, election, ministers, ...",2277,302
4,kilroy names election seat target ex chat show...,Politics,"[kilroy, names, election, seat, target, ex, ch...",1857,275


In [25]:
fig = px.histogram(data, x='words_count', marginal='violin')
fig.update_layout(width=1200, title="Word Count", template='plotly_dark')
fig.show()

In [26]:
fig = px.histogram(data_frame=data, x='length', marginal='violin')
fig.update_layout(width=1200, title="Word length", template='plotly_dark')
fig.show()

BOW (Bag Of Words)

In [28]:
# Import CountVectorizer BOW
from sklearn.feature_extraction.text import CountVectorizer

#Create CountVectorizer object
vectorizer = CountVectorizer(ngram_range=(1,2))

#Generate matrix of word vectors
count_vec = vectorizer.fit_transform(data["Text"])

# Print the shape of bo
count_vec.shape # The shape will be (n_samples, n_features)

(2225, 369424)

In [29]:
count_vec

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 825630 stored elements and shape (2225, 369424)>

In [30]:
# Create the CountVectorizer DataFrame: count_df
count_df = pd.DataFrame(count_vec.toarray(), columns=vectorizer.get_feature_names_out())
count_df.head()

,00,00 59,00 early,00 mark,00 per,00 qualifying,00 work,000,000 000,000 10,...,zurich reported,zutons,zutons 20,zvonareva,zvonareva lost,zvonareva russia,zvonareva struggled,zvonareva wimbledon,zvyagintsev,zvyagintsev return
0,0,0,0,0,0,0,0,5,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
# Import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

#Create CountVectorizer object
tdf_vectorizer = TfidfVectorizer()

# Genrate matrix of word vectors
tdf_vec = tdf_vectorizer.fit_transform(data['Text'])

# Print the shape of bo
tdf_vec.shape

(2225, 29276)

In [33]:
# Create the CountVectorizer DataFrame: count_df
tdf_df = pd.DataFrame(tdf_vec.toarray(), columns=tdf_vectorizer.get_feature_names_out())
tdf_df.head()

,00,000,0001,000bn,000m,000s,000th,001,001and,001st,...,zooms,zooropa,zornotza,zorro,zubair,zuluaga,zurich,zutons,zvonareva,zvyagintsev
0,0.0,0.104998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Prediction using Count Vectorizer


In [34]:
y=df['Label'].values

In [38]:
X=count_df

In [44]:
X = X.fillna("")
y = y.fillna(method='ffill')

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [46]:
from sklearn.metrics import log_loss, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

#  Convert text to numerical features
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# Get predicted probabilities
y_pred_prob = rf_model.predict_proba(X_test)

# Calculate log loss
logloss = log_loss(y_test, y_pred_prob)

print("Log loss value from RandomForest model =", logloss)

Log loss value from RandomForest model = 0.39287511398404595


In [52]:
acc_score = accuracy_score(y_test, y_pred)
acc_score

0.9348314606741573

In [56]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(rf_model,X_text,y_text,display_labels=("Politics","Sport","Technology","Entertainment","Business"),cmap="Blues")


NameError: name 'X_text' is not defined

In [57]:
def prediction(text:str):

  # Assuming 'vectorizer' is a pre-trained vectorizer
  word_vector = vectorizer.transform([text])

  # Assuming Random forest
  prediction_model = rf_model.predict(word_vector.astype(np.float64))

  return prediction_model

In [58]:
# Example Usage
example_text ="Nothing beats the joy of sharing pizza with friends while watching a baseball game on a sunny afternoon"
prediction_value = prediction(example_text)
print("Prediction:", prediction_value)

Prediction: ['Sport']
